In [ ]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import joblib
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from src.models.evaluator import (
    evaluate_model,
    print_classification_report,
    plot_confusion_matrix,
    plot_model_comparison
)
from src.models.reporter import (
    generate_classification_report_dict,
    generate_markdown_report,
    save_markdown_report,
    save_combined_metrics
)
from src.data.priority_labeler import (
    assign_priority_labels,
    PRIORITY_COLUMN
)
from src.utils.config import (
    MODELS_DIR,
    REPORTS_DIR,
    PROCESSED_DATA_DIR,
    CATEGORY_LABELS,
    PRIORITY_LABELS,
    CATEGORY_TARGET
)

sns.set_theme(style="whitegrid")
print("Imports successful")

In [ ]:
# Load models
cat_model = joblib.load(MODELS_DIR / "category_classifier.pkl")
pri_model = joblib.load(MODELS_DIR / "priority_classifier.pkl")
vectorizer = joblib.load(MODELS_DIR / "tfidf_vectorizer.pkl")

# Load splits
splits = joblib.load(MODELS_DIR / "train_test_splits.pkl")
X_test_tfidf = splits["X_test_tfidf"]
X_test_raw   = splits["X_test"]
y_cat_test   = splits["y_cat_test"]

# Load test data for priority labels
test_clean = pd.read_csv(PROCESSED_DATA_DIR / "test_cleaned.csv")
test_priority = assign_priority_labels(test_clean)
y_pri_test = test_priority[PRIORITY_COLUMN].reset_index(drop=True)

print("All artifacts loaded successfully")
print(f"Test set size: {len(y_cat_test)}")

In [ ]:
cat_metrics, cat_preds = evaluate_model(
    cat_model, X_test_tfidf, y_cat_test, "LinearSVC - Category"
)

print_classification_report(y_cat_test, cat_preds, "LinearSVC - Category")

plot_confusion_matrix(
    y_cat_test, cat_preds,
    "LinearSVC - Category",
    CATEGORY_LABELS,
    "final_confusion_matrix_category.png"
)

In [ ]:
pri_metrics, pri_preds = evaluate_model(
    pri_model, X_test_tfidf, y_pri_test, "LinearSVC - Priority"
)

print_classification_report(y_pri_test, pri_preds, "LinearSVC - Priority")

plot_confusion_matrix(
    y_pri_test, pri_preds,
    "LinearSVC - Priority",
    PRIORITY_LABELS,
    "final_confusion_matrix_priority.png"
)

In [ ]:
cat_report_dict = generate_classification_report_dict(
    y_cat_test, cat_preds, "category"
)

print("=== PER CLASS BREAKDOWN - CATEGORY ===\n")
print(f"{'Category':<25} {'Precision':>10} {'Recall':>10} {'F1':>10} {'Support':>10}")
print("-" * 65)

skip = {"accuracy", "macro avg", "weighted avg"}
for label, scores in cat_report_dict.items():
    if label in skip:
        continue
    if isinstance(scores, dict):
        print(
            f"{label:<25} "
            f"{scores['precision']:>10.3f} "
            f"{scores['recall']:>10.3f} "
            f"{scores['f1-score']:>10.3f} "
            f"{int(scores['support']):>10}"
        )

In [ ]:
pri_report_dict = generate_classification_report_dict(
    y_pri_test, pri_preds, "priority"
)

print("=== PER CLASS BREAKDOWN - PRIORITY ===\n")
print(f"{'Priority':<15} {'Precision':>10} {'Recall':>10} {'F1':>10} {'Support':>10}")
print("-" * 55)

for label, scores in pri_report_dict.items():
    if label in skip:
        continue
    if isinstance(scores, dict):
        print(
            f"{label:<15} "
            f"{scores['precision']:>10.3f} "
            f"{scores['recall']:>10.3f} "
            f"{scores['f1-score']:>10.3f} "
            f"{int(scores['support']):>10}"
        )

In [ ]:
summary = pd.DataFrame([
    {
        "Task":     "Category Classification",
        "Model":    "LinearSVC",
        "Classes":  7,
        "Accuracy": cat_metrics["accuracy"],
        "F1":       cat_metrics["f1_score"],
    },
    {
        "Task":     "Priority Prediction",
        "Model":    "LinearSVC",
        "Classes":  4,
        "Accuracy": pri_metrics["accuracy"],
        "F1":       pri_metrics["f1_score"],
    }
])

print("=== FINAL SYSTEM SUMMARY ===")
print(summary.to_string(index=False))

In [ ]:
from src.data.preprocessor import clean_text

print("=== LIVE PREDICTIONS ON UNSEEN TICKETS ===\n")

sample_indices = [0, 50, 100, 200, 300, 400, 500]

print(f"{'Ticket Text':<45} {'True Cat':<20} {'Pred Cat':<20} {'Priority'}")
print("-" * 105)

for idx in sample_indices:
    raw_text  = str(X_test_raw.iloc[idx])
    true_cat  = y_cat_test.iloc[idx]
    cleaned   = clean_text(raw_text)
    features  = vectorizer.transform([cleaned])
    pred_cat  = cat_model.predict(features)[0]
    pred_pri  = pri_model.predict(features)[0]
    match     = "✓" if pred_cat == true_cat else "✗"

    print(
        f"{raw_text[:43]:<45} ",
        f"{true_cat:<20} ",
        f"{pred_cat:<20} ",
        f"{pred_pri}  {match}"
    )

In [ ]:
report_md = generate_markdown_report(
    category_metrics=cat_metrics,
    priority_metrics=pri_metrics,
    category_report=cat_report_dict,
    priority_report=pri_report_dict,
    category_model_name="LinearSVC",
    priority_model_name="LinearSVC",
)

save_markdown_report(report_md, "final_report.md")
save_combined_metrics(cat_metrics, pri_metrics, "combined_metrics.json")

print("=== REPORT SAVED ===")
print("Location: outputs/reports/final_report.md")
print("Metrics:  outputs/reports/combined_metrics.json")

In [ ]:
import os

print("=== OUTPUT FILES VERIFICATION ===\n")

figures = [
    "category_distribution.png",
    "priority_distribution.png",
    "top_features_per_category.png",
    "text_length_before_after.png",
    "category_model_comparison.png",
    "priority_model_comparison.png",
    "final_confusion_matrix_category.png",
    "final_confusion_matrix_priority.png",
]

models = [
    "tfidf_vectorizer.pkl",
    "category_classifier.pkl",
    "priority_classifier.pkl",
    "train_test_splits.pkl",
]

reports = [
    "final_report.md",
    "combined_metrics.json",
    "category_metrics.json",
    "priority_metrics.json",
]

def check_files(folder, files):
    for f in files:
        path = folder / f
        status = "✅" if path.exists() else "❌ MISSING"
        print(f"  {status}  {f}")

from src.utils.config import FIGURES_DIR, MODELS_DIR, REPORTS_DIR

print("Figures:")
check_files(FIGURES_DIR, figures)

print("\nModels:")
check_files(MODELS_DIR, models)

print("\nReports:")
check_files(REPORTS_DIR, reports)